# Monte Carlo Finance Simulator — Demo

This notebook walks through the core pipeline:
1. Fetch real market data from Yahoo Finance
2. Run Monte Carlo simulations (GBM and bootstrap)
3. Compute risk metrics (VaR, CVaR, etc.)
4. Visualize results

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook'

from src.data_fetcher import fetch_historical, compute_log_returns, compute_drift_vol
from src.monte_carlo import gbm_simulation, simulate_with_returns, compute_percentiles
from src.analysis import summary_stats, probability_of_target
from src.visualization import build_dashboard_figures

## 1. Fetch Real Data

In [ ]:
TICKER = 'MSFT'

df = fetch_historical(TICKER, start='2020-01-01')
print(f"Fetched {len(df)} days of {TICKER} data")
df.tail()

## 2. Estimate Drift and Volatility

In [ ]:
log_returns, prices = compute_log_returns(df)
mu, sigma = compute_drift_vol(log_returns)
last_price = float(prices.iloc[-1])

print(f"Last price: ${last_price:.2f}")
print(f"Annualized drift (mu):   {mu:.4f} ({mu*100:.2f}%)")
print(f"Annualized volatility:   {sigma:.4f} ({sigma*100:.2f}%)")

## 3. Run Monte Carlo Simulation

In [ ]:
N_SIM = 2000
HORIZON = 252  # 1 trading year

paths_gbm = gbm_simulation(
    last_price=last_price,
    mu=mu,
    sigma=sigma,
    n_simulations=N_SIM,
    forecast_horizon=HORIZON,
)

print(f"GBM simulation shape: {paths_gbm.shape}")

## 4. Risk Metrics

In [ ]:
stats = summary_stats(paths_gbm, confidence=0.05)
for k, v in stats.items():
    print(f"  {k}: {v}")

target = last_price * 1.2
prob = probability_of_target(paths_gbm, target)
print(f"\nP(Price >= ${target:.2f}): {prob:.1%}")

## 5. Visualize

In [ ]:
percentiles = compute_percentiles(paths_gbm)
forecast_dates = pd.bdate_range(start=df.index[-1], periods=HORIZON + 1)

fig = build_dashboard_figures(
    df=df,
    paths=paths_gbm,
    percentiles=percentiles,
    forecast_dates=forecast_dates,
)
fig.show()

## 6. Compare with Bootstrap

In [ ]:
paths_boot = simulate_with_returns(
    last_price=last_price,
    log_returns=log_returns,
    n_simulations=N_SIM,
    forecast_horizon=HORIZON,
)

stats_boot = summary_stats(paths_boot, confidence=0.05)
print("Bootstrap metrics:")
for k, v in stats_boot.items():
    print(f"  {k}: {v}")